In [79]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [8]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:3]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .


In [16]:
X.shape, X.dtype, Y.shape, Y.dtype
# X: 16个样本，每个字符3个上下文

(torch.Size([16, 3]), torch.int64, torch.Size([16]), torch.int64)

In [18]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1]])

In [13]:
C = torch.randn((27,2))  # 2维，每个字母用2个参数表示

In [19]:
C

tensor([[ 0.1721, -0.8560],
        [ 0.3938, -0.1826],
        [ 0.5219, -1.0652],
        [ 1.9484, -1.1625],
        [ 0.2700,  0.9050],
        [-0.9389, -0.0909],
        [-0.5081,  1.5923],
        [ 2.1376,  0.8695],
        [-0.4335,  1.3086],
        [-0.7860, -0.3087],
        [-0.7993,  0.5530],
        [-0.2992, -0.3057],
        [ 1.5247,  0.3987],
        [ 0.3160, -0.6545],
        [-1.4681, -0.4267],
        [-0.6998,  1.3773],
        [-0.5181, -0.5805],
        [ 0.8123, -0.2642],
        [-0.0501, -0.3672],
        [ 0.1366,  1.7536],
        [ 0.1509, -0.6398],
        [ 1.1818, -0.6174],
        [ 0.0462, -1.1162],
        [ 0.8468,  2.0857],
        [ 0.7531,  1.1839],
        [ 1.0111,  0.4698],
        [ 0.6596,  0.2848]])

In [22]:
idx = torch.tensor([0, 2]) 
C[idx]

tensor([[ 0.1721, -0.8560],
        [ 0.5219, -1.0652]])

In [23]:
C[X].shape # 对X每个元素查表，再堆叠

torch.Size([16, 3, 2])

In [20]:
C[X] # 等价于查表

tensor([[[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560]],

        [[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.9389, -0.0909]],

        [[ 0.1721, -0.8560],
         [-0.9389, -0.0909],
         [ 0.3160, -0.6545]],

        [[-0.9389, -0.0909],
         [ 0.3160, -0.6545],
         [ 0.3160, -0.6545]],

        [[ 0.3160, -0.6545],
         [ 0.3160, -0.6545],
         [ 0.3938, -0.1826]],

        [[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560]],

        [[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.6998,  1.3773]],

        [[ 0.1721, -0.8560],
         [-0.6998,  1.3773],
         [ 1.5247,  0.3987]],

        [[-0.6998,  1.3773],
         [ 1.5247,  0.3987],
         [-0.7860, -0.3087]],

        [[ 1.5247,  0.3987],
         [-0.7860, -0.3087],
         [ 0.0462, -1.1162]],

        [[-0.7860, -0.3087],
         [ 0.0462, -1.1162],
         [-0.7860, -0.3087]],

        [[ 0.0462, -1

In [27]:
emb = C[X]
emb.shape

torch.Size([16, 3, 2])

In [36]:
# emb[sample, char_position, embedding_dim]                                          
#        ↑         ↑                ↑                                                  
#       dim0      dim1             dim2    
# emb[:, t, :] -> 所有样本的第 t 个字符
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1)

tensor([[ 0.1721, -0.8560,  0.1721, -0.8560,  0.1721, -0.8560],
        [ 0.1721, -0.8560,  0.1721, -0.8560, -0.9389, -0.0909],
        [ 0.1721, -0.8560, -0.9389, -0.0909,  0.3160, -0.6545],
        [-0.9389, -0.0909,  0.3160, -0.6545,  0.3160, -0.6545],
        [ 0.3160, -0.6545,  0.3160, -0.6545,  0.3938, -0.1826],
        [ 0.1721, -0.8560,  0.1721, -0.8560,  0.1721, -0.8560],
        [ 0.1721, -0.8560,  0.1721, -0.8560, -0.6998,  1.3773],
        [ 0.1721, -0.8560, -0.6998,  1.3773,  1.5247,  0.3987],
        [-0.6998,  1.3773,  1.5247,  0.3987, -0.7860, -0.3087],
        [ 1.5247,  0.3987, -0.7860, -0.3087,  0.0462, -1.1162],
        [-0.7860, -0.3087,  0.0462, -1.1162, -0.7860, -0.3087],
        [ 0.0462, -1.1162, -0.7860, -0.3087,  0.3938, -0.1826],
        [ 0.1721, -0.8560,  0.1721, -0.8560,  0.1721, -0.8560],
        [ 0.1721, -0.8560,  0.1721, -0.8560,  0.3938, -0.1826],
        [ 0.1721, -0.8560,  0.3938, -0.1826,  0.0462, -1.1162],
        [ 0.3938, -0.1826,  0.0462, -1.1

In [37]:
torch.unbind(emb, 1)

(tensor([[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.9389, -0.0909],
         [ 0.3160, -0.6545],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.6998,  1.3773],
         [ 1.5247,  0.3987],
         [-0.7860, -0.3087],
         [ 0.0462, -1.1162],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.3938, -0.1826]]),
 tensor([[ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.9389, -0.0909],
         [ 0.3160, -0.6545],
         [ 0.3160, -0.6545],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [-0.6998,  1.3773],
         [ 1.5247,  0.3987],
         [-0.7860, -0.3087],
         [ 0.0462, -1.1162],
         [-0.7860, -0.3087],
         [ 0.1721, -0.8560],
         [ 0.1721, -0.8560],
         [ 0.3938, -0.1826],
         [ 0.0462, -1.1162]]),
 tensor([[ 0.1721, -0.8560],
         [-0.9389, -0.0909],
         [

In [42]:
a = torch.arange(12)
a.shape

torch.Size([12])

In [43]:
a.view(3, 2, 2)

tensor([[[ 0,  1],
         [ 2,  3]],

        [[ 4,  5],
         [ 6,  7]],

        [[ 8,  9],
         [10, 11]]])

In [44]:
a.storage()

/var/folders/hb/8kn3wr9d1gn36n7n8nnc95cr0000gn/T/ipykernel_93471/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 12]

In [45]:
# torch底层存储的是一维，我们可以高效地用view来调整tensor shape
emb.shape

torch.Size([16, 3, 2])

In [46]:
emb.view(16, 6).shape

torch.Size([16, 6])

In [47]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [50]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # -1: 让torch自己推断
# emb 输入每个都是6维
# @ W1 -> 输出 100 维
h

tensor([[-0.8548, -0.9690,  0.9790,  ...,  0.9916, -0.0602,  0.3128],
        [-0.9674, -0.9740,  0.9891,  ...,  0.9872, -0.9564,  0.1383],
        [-0.1430,  0.0913,  0.9380,  ...,  0.5744, -0.0149, -0.1568],
        ...,
        [-0.7876, -0.9767,  0.9710,  ...,  0.9767,  0.5304, -0.2845],
        [-0.3808, -0.7557,  0.9441,  ...,  0.9765, -0.8737,  0.8702],
        [-0.8755, -0.9946,  0.9913,  ...,  0.9976,  0.5826,  0.0882]])

In [51]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [52]:
logits = h @ W2 + b2

In [54]:
logits.shape

torch.Size([16, 27])

In [65]:
counts = logits.exp() # exp 的导数就是它本身，方便计算
probs = counts / counts.sum(1, keepdims=True)

In [67]:
probs.shape # 16个样本，每个输入对应27个预测分布

torch.Size([16, 27])

In [59]:
probs[0].sum()

tensor(1.)

In [60]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0])

In [63]:
torch.arange(16)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [68]:
probs[torch.arange(16), Y]
# 每个索引管一个维度
# (arange(16)[0], Y[0]) = (0, Y[0])   →  probs[0, Y[0]]
# (arange(16)[1], Y[1]) = (1, Y[1])   →  probs[1, Y[1]]
# ...
# (arange(16)[15], Y[15]) = (15, Y[15]) →  probs[15, Y[15]]
# 拿到正确答案Y的概率

tensor([1.9751e-07, 1.7902e-03, 8.4770e-04, 1.1353e-04, 1.5197e-10, 1.7556e-06,
        2.0833e-09, 6.2581e-07, 4.4971e-01, 2.2158e-12, 1.1353e-06, 1.6043e-07,
        2.3307e-03, 5.0058e-12, 1.2783e-01, 7.9392e-09])

In [70]:
probs[torch.arange(16), Y].log() # log相乘便利性，方便计算mean

tensor([-15.4375,  -6.3254,  -7.0730,  -9.0835, -22.6073, -13.2527, -19.9893,
        -14.2842,  -0.7992, -26.8354, -13.6886, -15.6454,  -6.0616, -26.0204,
         -2.0571, -18.6515])

In [71]:
loss = -probs[torch.arange(16), Y].log().mean() # 平均惊讶度
loss

tensor(13.6133)

In [72]:
# make it clear
X.shape, Y.shape

(torch.Size([16, 3]), torch.Size([16]))

In [73]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator = g)
W1 = torch.randn((6, 100), generator = g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [74]:
sum(p.nelement() for p in parameters)

3481

In [81]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(16), Y].log().mean() 
loss

tensor(20.2417)

In [82]:
F.cross_entropy(logits, Y)

tensor(20.2417)